# Review Manhwa

Cắt panel, đọc chữ, viết lời review tiếng Việt, đọc bằng **VieNeu-TTS v3 Turbo**, rồi ghép video. Giao diện web hiện ngay trong notebook.

Google Drive gắn **cùng cách** với tool [Audio Full Truyện](https://github.com/hiepit127/generate-ytb-audio): Colab gắn Drive thành ổ đĩa, `colab/setup_colab.py` tạo thư mục và gán biến môi trường. App ghi file bằng đường dẫn thường. Drive tự đồng bộ. Không dùng API Google Drive, không có file credential trong repo.

### Chuẩn bị (làm 1 lần mỗi phiên)

1. **Bật GPU:** `Runtime → Change runtime type → T4 GPU` → Save.
2. **Repo private:** biểu tượng 🔑 bên trái → thêm secret đúng tên `GITHUB_TOKEN` (fine-grained, chỉ repo này, quyền **Contents: Read-only**) và bật **Notebook access**. Token không ghi vào notebook.
3. Bấm **Runtime → Run all**.

Ô 2 gắn Drive. Dữ liệu nằm ở `MyDrive/ReviewManhwa/` (đổi bằng ô `DRIVE_DIR`):

| Thư mục | Việc |
|---|---|
| `cache/jobs` | Panel, chữ OCR, audio, việc đang làm |
| `cache/voices` | Giọng đã clone |
| `output` | Video mp4, phụ đề srt, wav đọc thử |
| `hf-cache` | Trọng số VieNeu, để lần sau khỏi tải lại |

Bỏ chọn `MODEL_ON_DRIVE` nếu Drive gần đầy: model nằm ở ổ local của phiên (`/content/hf-cache`). Chưa gắn được Drive, hoặc thư mục không ghi được, thì tool lưu vào `cache/` và `output/` cạnh mã nguồn.

Chỉ tải lên truyện bạn có quyền sử dụng. Tool không lấy chương từ website. API key viết lời nhập trong giao diện, chỉ nằm trong RAM của phiên, không ghi lên Drive.


In [ ]:
#@title 1️⃣ Kiểm tra GPU
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
        capture_output=True, text=True,
    ).stdout)
    print("✅ Có GPU.")
else:
    print("⚠️  KHÔNG có GPU. Vào Runtime → Change runtime type → T4 GPU, rồi Runtime → Run all.")
    print("   Vẫn chạy được trên CPU nhưng OCR và VieNeu sẽ chậm.")


In [ ]:
#@title 2️⃣ Cấu hình + kết nối Google Drive
DRIVE_DIR = "/content/drive/MyDrive/ReviewManhwa"  #@param {type:"string"}
GITHUB_REPO = "hiepit127/review-manhwa"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
MODEL_ON_DRIVE = True  #@param {type:"boolean"}
#@markdown `MODEL_ON_DRIVE`: lưu trọng số VieNeu vào `hf-cache` trên Drive. Bỏ chọn nếu Drive gần đầy.

import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("✅ Dữ liệu sẽ lưu tại:", DRIVE_DIR)


In [ ]:
#@title 3️⃣ Tải mã nguồn từ GitHub
import os, re, base64, subprocess

REPO_DIR = "/content/review-manhwa"

def _read_secret(name):
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return os.environ.get(name, "").strip()

def _token():
    # Colab đôi khi cắt tên secret. Vẫn ưu tiên đúng tên GITHUB_TOKEN.
    for name in ("GITHUB_TOKEN", "GITHUB_TOKEN1", "GH_TOKEN", "GITHUB_PAT"):
        value = _read_secret(name)
        if value.lower().startswith("bearer "):
            value = value.split(None, 1)[1].strip()
        if value:
            print(f"Đã đọc secret {name}.")
            return value
    print("Không thấy secret GITHUB_TOKEN. Repo public thì vẫn clone được.")
    return ""

def _redact(text, token):
    cleaned = text or ""
    if token:
        cleaned = cleaned.replace(token, "***")
    cleaned = re.sub(r"x-access-token:[^@\s]+", "x-access-token:***", cleaned)
    cleaned = re.sub(r"ghp_[A-Za-z0-9]+", "ghp_***", cleaned)
    cleaned = re.sub(r"github_pat_[A-Za-z0-9_]+", "github_pat_***", cleaned)
    return cleaned.strip()

token = _token()
git = ["git"]
if token:
    # Header cho riêng lệnh này. Không ghi vào .git/config và không đưa token lên URL.
    auth = base64.b64encode(f"x-access-token:{token.strip()}".encode()).decode()
    git += ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}"]
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
env.pop("GITHUB_TOKEN", None)
if os.path.isdir(f"{REPO_DIR}/.git"):
    result = subprocess.run(
        git + ["-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH],
        capture_output=True, text=True, env=env,
    )
else:
    result = subprocess.run(
        git + ["clone", "--depth", "1", "-b", BRANCH, f"https://github.com/{GITHUB_REPO}.git", REPO_DIR],
        capture_output=True, text=True, env=env,
    )
print(_redact((result.stdout or "") + (result.stderr or ""), token))
if result.returncode != 0:
    raise SystemExit(
        "Không tải được repo.\n"
        "Repo private: thêm secret GITHUB_TOKEN (Contents: Read-only) và bật Notebook access, rồi chạy lại ô này.\n"
        "Kiểm tra GITHUB_REPO và BRANCH ở ô 2."
    )
os.chdir(REPO_DIR)
print("✅ Mã nguồn tại", REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)


In [ ]:
#@title 4️⃣ Tạo thư mục trên Drive + cài thư viện
import importlib, sys
sys.path.insert(0, f"{REPO_DIR}/colab")
sys.path.insert(0, REPO_DIR)
import setup_colab as sc
importlib.reload(sc)
sc.prepare(DRIVE_DIR, hf_on_drive=MODEL_ON_DRIVE)
sc.install()
sc.doctor()


In [ ]:
#@title 5️⃣ Mở giao diện
import importlib, os, sys
sys.path.insert(0, f"{REPO_DIR}/colab")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
import setup_colab as sc
importlib.reload(sc)
sc.prepare(DRIVE_DIR, hf_on_drive=MODEL_ON_DRIVE)
sc.close_ports()
import app
import src.storage
importlib.reload(src.storage)
importlib.reload(app)
print(src.storage.summary_lines())
app.launch()


## Cách dùng

Thanh trên cùng hiện đang lưu ở đâu. Làm lần lượt bốn bước:

1. **Nguồn** — tải ảnh, PDF hoặc ZIP, hoặc dán từng trang rồi bấm **Thêm trang này**. Sau đó **Tách panel**. Việc cũ nằm ở **Việc đã lưu**.
2. **Kịch bản** — dán API key xAI (tạo tại [console.x.ai](https://console.x.ai)), bấm **Viết kịch bản**. Không có API thì **Chia cảnh, tôi viết tay**.
3. **Giọng** — chọn giọng có sẵn hoặc giọng đã clone, rồi **Nghe giọng này**.
4. **Video** — **Nghe thử câu kịch bản**, rồi **Xuất video**. File được chép vào `output/`.

Màn **Tạo giọng** là chỗ clone: tải file hoặc ghi âm, rồi **Tạo giọng**. Giọng nằm ở `cache/voices`. Bước **Giọng** và màn **Đọc** chỉ chọn giọng đã có.

Giữ tab Colab mở khi đang dùng. Phiên miễn phí tự ngắt khi để không hoặc sau tối đa khoảng 12 giờ. Chạy lại **Runtime → Run all**: dữ liệu trên Drive còn, link Gradio sẽ đổi. Muốn tắt giao diện thì bấm dừng ô đang chạy, rồi chạy lại ô 5.
